# Credit Risk Scorecard — Data Cleaning
**Author:** Nico Avila
**Date:** March 2026
**Input:** application_train.csv (307,511 rows · 122 columns)
**Output:** cleaned_data.csv

## Objective
Clean the raw Home Credit dataset for feature engineering and modelling.
This notebook covers missing value treatment, anomaly correction, and
column dropping — preserving all transformations for full reproducibility.
No new features are engineered here; that work lives in 03_features.ipynb.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r'C:\Users\U\Desktop\CreditRisk\application_train.csv')
print(f"Raw data loaded: {df.shape[0]:,} rows · {df.shape[1]} columns")

Raw data loaded: 307,511 rows · 122 columns


## Section 1 — Missing value analysis

Before dropping or imputing anything, we first quantify missingness across
all 122 columns. Columns above 60% missing will be dropped entirely —
they carry insufficient information to be useful predictors and would
introduce noise into the model.

In [2]:
# Calculate missingness for every column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# Split into above and below threshold
threshold = 60
high_missing = missing_df[missing_df['missing_pct'] > threshold]
low_missing = missing_df[(missing_df['missing_pct'] > 0) & 
                          (missing_df['missing_pct'] <= threshold)]

print(f"Columns above {threshold}% missing: {len(high_missing)}")
print(f"Columns below {threshold}% missing: {len(low_missing)}")
print(f"Columns with no missing values: {(missing_df['missing_pct'] == 0).sum()}")
print(f"\nTop 15 highest missingness columns:")
print(high_missing.head(15))

Columns above 60% missing: 17
Columns below 60% missing: 47
Columns with no missing values: 58

Top 15 highest missingness columns:
                          missing_count  missing_pct
COMMONAREA_MEDI                  214865         69.9
COMMONAREA_AVG                   214865         69.9
COMMONAREA_MODE                  214865         69.9
NONLIVINGAPARTMENTS_MODE         213514         69.4
NONLIVINGAPARTMENTS_AVG          213514         69.4
NONLIVINGAPARTMENTS_MEDI         213514         69.4
LIVINGAPARTMENTS_MODE            210199         68.4
LIVINGAPARTMENTS_AVG             210199         68.4
FONDKAPREMONT_MODE               210295         68.4
LIVINGAPARTMENTS_MEDI            210199         68.4
FLOORSMIN_AVG                    208642         67.8
FLOORSMIN_MODE                   208642         67.8
FLOORSMIN_MEDI                   208642         67.8
YEARS_BUILD_MEDI                 204488         66.5
YEARS_BUILD_MODE                 204488         66.5


## Section 2 — Drop high-missingness columns

Columns exceeding 60% missingness are primarily building and property
characteristics (COMMONAREA, NONLIVINGAPARTMENTS, FLOORSMIN etc.).
These were identified in the EDA as having 70%+ missing rates and
no reliable imputation strategy given the scale of absence.

**Decision:** Drop all columns above 60% missing threshold.
**Rationale:** Imputing 200,000+ missing values in a single column
would introduce more noise than signal into the model.

In [3]:
# Drop columns above 60% missing
cols_to_drop = high_missing.index.tolist()
df_cleaned = df.drop(columns=cols_to_drop)

print(f"Columns dropped: {len(cols_to_drop)}")
print(f"Remaining shape: {df_cleaned.shape}")
print(f"\nDropped columns:")
for col in cols_to_drop:
    print(f"  {col}: {missing_df.loc[col, 'missing_pct']}% missing")

Columns dropped: 17
Remaining shape: (307511, 105)

Dropped columns:
  COMMONAREA_MEDI: 69.9% missing
  COMMONAREA_AVG: 69.9% missing
  COMMONAREA_MODE: 69.9% missing
  NONLIVINGAPARTMENTS_MODE: 69.4% missing
  NONLIVINGAPARTMENTS_AVG: 69.4% missing
  NONLIVINGAPARTMENTS_MEDI: 69.4% missing
  LIVINGAPARTMENTS_MODE: 68.4% missing
  LIVINGAPARTMENTS_AVG: 68.4% missing
  FONDKAPREMONT_MODE: 68.4% missing
  LIVINGAPARTMENTS_MEDI: 68.4% missing
  FLOORSMIN_AVG: 67.8% missing
  FLOORSMIN_MODE: 67.8% missing
  FLOORSMIN_MEDI: 67.8% missing
  YEARS_BUILD_MEDI: 66.5% missing
  YEARS_BUILD_MODE: 66.5% missing
  YEARS_BUILD_AVG: 66.5% missing
  OWN_CAR_AGE: 66.0% missing


## Section 3 — Fix the DAYS_EMPLOYED anomaly

The DAYS_EMPLOYED column contains a data entry anomaly: the value 365,243
was used to represent unemployed or pensioner applicants (i.e. not applicable).
This is not a real employment duration — it represents approximately 1,000 years.

**Impact if uncorrected:** EMPLOYMENT_YEARS and EMPLOYED_TO_AGE_RATIO features
would be severely distorted for ~55,000 applicants.

**Fix:** Replace 365,243 with 0 and create a binary flag DAYS_EMPLOYED_ANOM
to preserve the information that these applicants are unemployed or pensioners.
This flag is itself a useful risk signal.

In [4]:
# Identify and fix the 365,243 anomaly
anomaly_count = (df_cleaned['DAYS_EMPLOYED'] == 365243).sum()
print(f"Anomaly records found: {anomaly_count:,}")
print(f"This represents {anomaly_count/len(df_cleaned)*100:.1f}% of applicants")

# Create binary flag before replacing
df_cleaned['DAYS_EMPLOYED_ANOM'] = (df_cleaned['DAYS_EMPLOYED'] == 365243).astype(int)

# Replace anomaly with 0
df_cleaned['DAYS_EMPLOYED'] = df_cleaned['DAYS_EMPLOYED'].replace(365243, 0)

# Verify fix
print(f"\nAnomaly records after fix: {(df_cleaned['DAYS_EMPLOYED'] == 365243).sum()}")
print(f"DAYS_EMPLOYED_ANOM flag created: {df_cleaned['DAYS_EMPLOYED_ANOM'].sum():,} records flagged")
print(f"\nDefault rate for flagged applicants: "
      f"{df_cleaned[df_cleaned['DAYS_EMPLOYED_ANOM']==1]['TARGET'].mean():.1%}")
print(f"Default rate for non-flagged applicants: "
      f"{df_cleaned[df_cleaned['DAYS_EMPLOYED_ANOM']==0]['TARGET'].mean():.1%}")

Anomaly records found: 55,374
This represents 18.0% of applicants

Anomaly records after fix: 0
DAYS_EMPLOYED_ANOM flag created: 55,374 records flagged

Default rate for flagged applicants: 5.4%
Default rate for non-flagged applicants: 8.7%


## Section 4 — Impute remaining missing values

For columns with less than 60% missingness, we apply a simple but defensible
imputation strategy:

- **Numerical columns** — impute with the **median** (robust to outliers)
- **Categorical columns** — impute with the **mode** (most frequent category)

This is consistent with the imputation strategy documented in the BRD
Risk Register (FR-05) and is standard practice for credit scorecard development.

In [5]:
# Separate numerical and categorical columns
numerical_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()

# Impute numerical with median
num_missing_before = df_cleaned[numerical_cols].isnull().sum().sum()
for col in numerical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Impute categorical with mode
cat_missing_before = df_cleaned[categorical_cols].isnull().sum().sum()
for col in categorical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col].fillna(df_cleaned[col].mode()[0], inplace=True)

# Verify
total_missing_after = df_cleaned.isnull().sum().sum()
print(f"Numerical missing values imputed: {num_missing_before:,}")
print(f"Categorical missing values imputed: {cat_missing_before:,}")
print(f"Total missing values remaining: {total_missing_after}")
print(f"\nFinal cleaned shape: {df_cleaned.shape}")

C:\Users\U\AppData\Local\Temp\ipykernel_1304\156243436.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(df_cleaned[col].mode()[0], inplace=True)


Numerical missing values imputed: 5,030,041
Categorical missing values imputed: 554,076
Total missing values remaining: 0

Final cleaned shape: (307511, 106)


## Section 5 — Outlier treatment

Two columns require outlier capping based on EDA findings:

- **CNT_CHILDREN** — values above 5 represent fewer than 10 records
  with 100% default rates, statistically unreliable. Capped at 5.
- **AMT_INCOME_TOTAL** — extreme values above the 99.5th percentile
  are capped to reduce the influence of ultra-high-income outliers.

In [6]:
# Cap CNT_CHILDREN at 4 (EDA finding — values above 4 are statistically unreliable)
df_cleaned['CNT_CHILDREN'] = df_cleaned['CNT_CHILDREN'].clip(upper=4)
print(f"CNT_CHILDREN capped at 4")
print(f"Distribution after cap:\n{df_cleaned['CNT_CHILDREN'].value_counts().sort_index()}")

# Cap AMT_INCOME_TOTAL at 99.5th percentile
income_cap = df_cleaned['AMT_INCOME_TOTAL'].quantile(0.995)
df_cleaned['AMT_INCOME_TOTAL'] = df_cleaned['AMT_INCOME_TOTAL'].clip(upper=income_cap)
print(f"\nAMT_INCOME_TOTAL capped at 99.5th percentile: {income_cap:,.0f}")

CNT_CHILDREN capped at 4
Distribution after cap:
CNT_CHILDREN
0    215371
1     61119
2     26749
3      3717
4       555
Name: count, dtype: int64

AMT_INCOME_TOTAL capped at 99.5th percentile: 630,000


## Section 6 — Cleaning validation

Final checks before saving the cleaned dataset:
- Zero missing values remaining
- Shape is consistent with expectations
- Target distribution is unchanged from raw data
- Key columns are within expected ranges

In [7]:
print("=== CLEANING VALIDATION ===\n")

# 1. Missing values check
total_missing = df_cleaned.isnull().sum().sum()
print(f"Missing values remaining: {total_missing} {'✓' if total_missing == 0 else '✗ — check imputation'}")

# 2. Shape check
print(f"Final shape: {df_cleaned.shape[0]:,} rows · {df_cleaned.shape[1]} columns")

# 3. Target distribution unchanged
print(f"\nTarget distribution (must match raw 8.1%):")
print(f"Default rate: {df_cleaned['TARGET'].mean():.1%} {'✓' if abs(df_cleaned['TARGET'].mean() - 0.081) < 0.001 else '✗'}")

# 4. DAYS_EMPLOYED anomaly resolved
max_employed = df_cleaned['DAYS_EMPLOYED'].max()
print(f"\nDAYS_EMPLOYED max value: {max_employed} {'✓' if max_employed < 365243 else '✗ — anomaly not fixed'}")

# 5. New columns added
print(f"\nNew columns added during cleaning:")
new_cols = ['DAYS_EMPLOYED_ANOM']
for col in new_cols:
    print(f"  {col}: {df_cleaned[col].sum():,} records")

print(f"\nCleaning validation complete.")

=== CLEANING VALIDATION ===

Missing values remaining: 0 ✓
Final shape: 307,511 rows · 106 columns

Target distribution (must match raw 8.1%):
Default rate: 8.1% ✓

DAYS_EMPLOYED max value: 0 ✓

New columns added during cleaning:
  DAYS_EMPLOYED_ANOM: 55,374 records

Cleaning validation complete.


In [8]:
print("=== CLEANING SUMMARY ===")
summary = {
    'Raw shape':           '307,511 rows · 122 columns',
    'Columns dropped':     '17 (>60% missing)',
    'Anomaly records fixed': '55,374 (DAYS_EMPLOYED)',
    'Values imputed':      '5,584,117',
    'Final shape':         '307,511 rows · 106 columns',
    'Missing values':      '0',
}
for k, v in summary.items():
    print(f"  {k:<30} {v}")

=== CLEANING SUMMARY ===
  Raw shape                      307,511 rows · 122 columns
  Columns dropped                17 (>60% missing)
  Anomaly records fixed          55,374 (DAYS_EMPLOYED)
  Values imputed                 5,584,117
  Final shape                    307,511 rows · 106 columns
  Missing values                 0


In [10]:
df_cleaned.to_csv(r'C:\Users\U\Desktop\CreditRisk\cleaned_data.csv', index=False)
print(f"Saved: cleaned_data.csv — {df_cleaned.shape[0]:,} rows · {df_cleaned.shape[1]} columns")

Saved: cleaned_data.csv — 307,511 rows · 106 columns
